# GNN time
### Disclaimer: This is a training exercise, I understand that I'm using a model to predict something that is completely fabricated by me


In [ ]:
import torch    
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from collections import Counter
from torch_geometric.data import InMemoryDataset, Data
import torch_geometric
from torch_geometric.nn import GATv2Conv
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



In [ ]:
# Load the data
nodes_gnn = gpd.read_file('nodes_PC.geojson')
edges_gnn = gpd.read_file('edges_PC.geojson')

In [ ]:
# Check the data
nodes_gnn.head()

In [ ]:
# Check the data
f, ax = plt.subplots(figsize=(8,8))
nodes_gnn.plot(ax=ax, color='blue', markersize=3, zorder=2)
edges_gnn.plot(ax=ax, linewidth= 1, edgecolor= 'green', zorder=1)

In [ ]:
# Preprocess the data and prepare the dataset

X = nodes_gnn.drop(columns=['geometry', 'railway', 'highway'])
X = np.array(X.values)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

In [ ]:
edges_gnn.columns

In [ ]:
nodes_gnn['osmid']

In [ ]:
Y = edges_gnn['traffic_speed'].values
Y = np.array(Y)
# Prepare edge index 
node_to_id = {}
for i, node in enumerate(nodes_gnn['osmid'].values):
    node_to_id[node] = i

start_node = [node_to_id[i] for i in edges_gnn['u'].values]
end_node = [node_to_id[i] for i in edges_gnn['v'].values]
start = torch.tensor(start_node, dtype=torch.long)
end = torch.tensor(end_node, dtype=torch.long)
edge_index = torch.stack([start, end], dim=0)

In [ ]:
# Create dataset for edge regression
class CremonaDataset(InMemoryDataset):
    def __init__(self, edge_index, x, y, pre_transform = None , transform=None):
        super(CremonaDataset, self).__init__('.', transform, None, None)
        
        # Create data object
        data = Data(x=torch.FloatTensor(x), 
                   y=torch.FloatTensor(y), 
                   edge_index=edge_index)
        
        # Set number of nodes and features
        data.num_nodes = x.shape[0]
        data.num_features = x.shape[1]
        data.num_edges = edge_index.shape[1]
        
        # Split data into train/val/test sets (70%/15%/15%)
        np.random.seed(0)
        n_nodes = data.num_nodes
        n_edges = data.num_edges
        indices = np.random.permutation(n_edges)
        #print(n_nodes) #debug
        
        train_idx = indices[:int(0.7 * n_edges)]
        #print(len(train_idx)) #debug
        val_idx = indices[int(0.7 * n_edges):int(0.85 * n_edges)]
        test_idx = indices[int(0.85 * n_edges):]
        
        # Create masks
        train_mask = torch.zeros(n_edges, dtype=torch.bool)
        val_mask = torch.zeros(n_edges, dtype=torch.bool)
        test_mask = torch.zeros(n_edges, dtype=torch.bool)
        
        train_mask[train_idx] = True
        val_mask[val_idx] = True
        test_mask[test_idx] = True
        
        data.train_mask = train_mask    #size is still all nodes but only True for train nodes
        data.val_mask = val_mask
        data.test_mask = test_mask
        
        self.data, self.slices = self.collate([data])
        
        print("Dataset created with {} nodes".format(data.num_nodes), "and {} edges".format(edge_index.shape[1]))
    
    
# Set torch device to cuda if available or else cpu
#device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

dataset = CremonaDataset(edge_index, X_scaled, Y)
data = dataset[0].to(device)

In [ ]:
data.train_mask 

In [ ]:
# Define hyperparameters
HIDDEN_DIM = 64
LR = 0.01


class GATv2(torch.nn.Module):
    def __init__(self, hidden_layer, data, output_dim = 1 , heads=2):
        super(GATv2, self).__init__()
        self.gat1 = GATv2Conv(data.num_features, hidden_layer, heads=heads)
        self.gat2 = GATv2Conv(hidden_layer*heads, hidden_layer, heads=1)
        self.linear = nn.Linear(hidden_layer*2, output_dim)
    
    def forward(self, data):
        # Get node features and edge index
        x, edge_index = data.x, data.edge_index
        
        # First layer
        out= self.gat1(x, edge_index)
        out = F.elu(out)
        out = F.dropout(out, p= 0.5, training=self.training)
        
        # Second layer
        out = self.gat2(out, edge_index)
        
        src, dst = edge_index   # src and dst are the start and end nodes of the edges
        out = torch.concat((out[src], out[dst]), dim=1)  # Concatenate the features of the start and end nodes
        
        # Final layer
        out = self.linear(out)

        return out
    
    def forward_attention(self, data):
        # Get node features and edge index
        x, edge_index = data.x, data.edge_index

        # First layer with attention weights
        out, (edge_index_1, attention_weights_1) = self.gat1(x, edge_index, return_attention_weights=True)
        out = F.elu(out)
        out = F.dropout(out, p=0.5, training=self.training)

        # Second layer with attention weights
        out, (edge_index_2, attention_weights_2) = self.gat2(out, edge_index, return_attention_weights=True)

        # Continue with your existing logic...
        src, dst = edge_index
        out = torch.concat((out[src], out[dst]), dim=1)
        out = self.linear(out)

        # Return output and attention weights if needed
        return out, (edge_index_1, attention_weights_1), (edge_index_2, attention_weights_2)

        


# Initialise GCN model instance
model = GATv2(HIDDEN_DIM, data).to(device) 
print(model)

In [ ]:
# Define loss and optimizer
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

# Training function
def train():
    model.train()
    optimizer.zero_grad()
    # Forward pass
    out = model(data)
    # Compute loss (only on training nodes)
    loss = criterion(out[data.train_mask].squeeze(), data.y[data.train_mask]) # squeeze() to remove extra dimension
    # Backward pass
    loss.backward()
    optimizer.step()
    return loss.item()

# Evaluation function
def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out[mask].squeeze()
        true = data.y[mask]
        loss = criterion(pred, true)
    return loss.item()

In [ ]:
out, (edge_index_1, attention_weights_1), (edge_index_2, attention_weights_2) = model.forward_attention(data)
edge_index_1.shape, attention_weights_1.shape



In [ ]:
import networkx as nx

import matplotlib.pyplot as plt

# Create a directed graph from edge_index_1
G = nx.DiGraph()
edges = [(edge_index_1[0, i].item(), edge_index_1[1, i].item()) for i in range(edge_index_1.shape[1])]
G.add_edges_from(edges)

# Get positions for nodes using the geometry from nodes_gnn
pos = {i: (row.geometry.x, row.geometry.y) for i, row in nodes_gnn.iterrows()}

# Create figure
plt.figure(figsize=(12, 10))

# Plot the network
nx.draw_networkx(G, pos, 
                 node_size=5,
                 node_color='blue',
                 edge_color='green',
                 width=0.3,
                 arrows=False,
                 with_labels=False)

plt.title('Graph Structure from edge_index_1')
plt.axis('off')
plt.tight_layout()
plt.show()

IMPORTANT STAFF I LEARNED: ATTENTION WEIGHTS ARE OF NUMBER EDGES + NUMBER NODES - SELF LOOPS, this is in order to not count the self loops twice.
KEEP GOING FROM HERE


## Let's study a given node and its neighbors
Informations flow from source to target along the edges, neighbors that have a role in computing the attention are only those that "flow" into a given node.

In [ ]:
# find the node with the most neighbors
degree_dict = dict(G.degree())
max_degree_node = max(degree_dict, key=degree_dict.get)
max_degree = degree_dict[max_degree_node]
print(f"Node with most neighbors: {max_degree_node} with degree {max_degree}")

In [ ]:
import seaborn as sns
# Study only one node
node_id = 11
# find nodes that point to node_id
target_mask = edge_index_1[1] == node_id
neighbors = edge_index_1[0][target_mask].cpu().detach().numpy()

print(f"Node {node_id} has {len(neighbors)} neighbors")
print(f"Neighbors: {neighbors}")

# Get the attention weights for the edges pointing to node_id
attention_weights = attention_weights_1[target_mask].cpu().detach().numpy()
attention_weights1 = attention_weights[:, 0]  # Select only one head
attention_weights2 = attention_weights[:, 1]  # Select only one head
print(f"Attention weights for edges pointing to node {node_id}: {attention_weights}")
print(f"Total attention weight: {np.sum(attention_weights)}") #2 heads so should be 2

In [ ]:
# plot selected node
f, ax = plt.subplots(figsize=(8,8))
ax.scatter(nodes_gnn.x[node_id], nodes_gnn.y[node_id], color='red', s=100, label='Selected Node', zorder=3)
ax.scatter(nodes_gnn.x[neighbors], nodes_gnn.y[neighbors], color='blue', s=100, label='Neighbors', zorder=2)
edges_gnn.plot(ax= ax, linewidth= 1, edgecolor= 'green', zorder=1)

#set xlim and ylim around the selected node
zoom = 100
ax.set_xlim(nodes_gnn.x[node_id] - zoom, nodes_gnn.x[node_id] + zoom)
ax.set_ylim(nodes_gnn.y[node_id] - zoom, nodes_gnn.y[node_id] + zoom)
ax.set_title(f"Node {node_id} and its neighbors")
ax.set_axis_off()

In [ ]:
edge_mask = edges_gnn['v'] == nodes_gnn['osmid'][node_id]
edges = edges_gnn[edge_mask]
edges = edges.reset_index(drop=True)
edges['Attention Score1'] = attention_weights1[0:len(edges)]
edges['Attention Score2'] = attention_weights2[0:len(edges)]



#plot 
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(16,8))
edges.plot(ax=ax1, column='Attention Score1', cmap='viridis', legend=True, linewidth=1)
ax1.scatter(nodes_gnn.x[node_id], nodes_gnn.y[node_id], color='red', s=100, label='Selected Node', zorder=3)
ax1.scatter(nodes_gnn.x[neighbors], nodes_gnn.y[neighbors], color='blue', s=100, label='Neighbors', zorder=2)
ax1.set_title('Attention Score1')
ax1.set_axis_off()

edges.plot(ax=ax2, column='Attention Score2', cmap='viridis', legend=True, linewidth=1)
ax2.scatter(nodes_gnn.x[node_id], nodes_gnn.y[node_id], color='red', s=100, label='Selected Node', zorder=3)
ax2.scatter(nodes_gnn.x[neighbors], nodes_gnn.y[neighbors], color='blue', s=100, label='Neighbors', zorder=2)
ax2.set_title('Attention Score2')
ax2.set_axis_off()




In [ ]:
# Training loop
EPOCHS = 1000
best_loss = float('inf')
early_stop_counter = 0
early_stop_patience = 30
train_losses, val_losses = [], []

print(f"Starting training on {device}...")
for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss = train()
    train_losses.append(train_loss)
    
    # Evaluate on validation set
    val_loss = evaluate(data.val_mask)
    val_losses.append(val_loss)
    
    # Adjust learning rate
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_CR_gat.pt')
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        
        
    
    # Print progress
    if epoch % 10 == 0:
        test_loss  = evaluate(data.test_mask)
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Test Loss: {test_loss:.4f}, ')
        
    # Check for early stopping
    if early_stop_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}")
        break

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()

In [ ]:
# Load best model for final evaluation
model = GATv2(HIDDEN_DIM, data).to(device)
model.load_state_dict(torch.load('best_CR_gat.pt'))
test_loss = evaluate(data.test_mask)
print(f'Final Test Results - Loss: {test_loss:.4f}')

In [ ]:
# Generate predictions for all nodes
model.eval()
with torch.no_grad():
    predictions = model(data).squeeze().cpu().numpy()

# Normalize the predictions
scaler = MinMaxScaler()
predictions = scaler.fit_transform(predictions.reshape(-1, 1))
    
# Add predictions back to the geodataframe for visualization
edges_gnn['gat_predictions'] = predictions


#Visualize the predictions

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot the original traffic speed
edges_gnn.plot(column='traffic_speed', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('Original Traffic Speed')
ax1.set_axis_off()

# Plot the predicted traffic speed
edges_gnn.plot(column='gat_predictions', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('Predicted Traffic Speed')
ax2.set_axis_off()

plt.tight_layout()
plt.show()

# Next steps I would like to try a Graph Transformer and see how it works
## Instead of a GNN, let's try a Graph Transformer
Let's start with TransformerConv, which should still be very similar to GNNs since its attention mechanism is based on the edges

In [ ]:
# Define hyperparameters
HIDDEN_DIM = 64
LR = 0.01
EPOCHS = 1000

class GT(torch.nn.Module):
    def __init__(self, hidden_layer, data, output_dim = 1 , heads=4):
        super(GT, self).__init__()
        self.transf1 = torch_geometric.nn.TransformerConv(data.num_features, hidden_layer, heads=heads, concat=True, dropout=0.5)
        self.layernorm1 = torch.nn.LayerNorm(hidden_layer*heads)
        
        self.transf2 = torch_geometric.nn.TransformerConv(hidden_layer*heads, hidden_layer, heads= heads, concat=False) 
        # No concat here because it is the last layer, which means that it will average the heads
        
        
        self.linear = nn.Linear(hidden_layer*2, output_dim)

    def forward(self, data):
        # Get node features and edge index
        x, edge_index = data.x, data.edge_index
        
        # First layer
        out = self.transf1(x, edge_index)
        out = self.layernorm1(out)
        out = F.relu(out)
        
        # Second layer
        out = self.transf2(out, edge_index) # No Norm and activation function because it is the last layer
        
        
        src, dst = edge_index   # src and dst are the start and end nodes of the edges
        out = torch.concat((out[src], out[dst]), dim=1)  # Concatenate the features of the start and end nodes
        
        # Final layer
        out = self.linear(out)    
        return out
    
    def reset_parameters(self):
        self.transf1.reset_parameters()
        self.transf2.reset_parameters()
        self.layernorm1.reset_parameters()
        
# Initialise GT model instance
model = GT(HIDDEN_DIM, data).to(device)
print(model)

In [ ]:
# Define loss and optimizer
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)


def train_tr():
    model.train()
    optimizer.zero_grad()
    # Forward pass
    out = model(data)
    # Compute loss (only on training nodes)
    loss = criterion(out[data.train_mask].squeeze(), data.y[data.train_mask]) # squeeze() to remove extra dimension
    # Backward pass
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate_tr(mask):
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out[mask].squeeze()
        true = data.y[mask]
        loss = criterion(pred, true)
    return loss.item()

In [ ]:
out = model(data)
out[data.train_mask].squeeze()

In [ ]:
# Training loop
best_loss = float('inf')
early_stop_counter = 0
early_stop_patience = 30
train_losses, val_losses = [], []


print(f"Starting training on {device}...")
model.reset_parameters()
for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss = train_tr()
    train_losses.append(train_loss)
    
    # Evaluate on validation set
    val_loss = evaluate_tr(data.val_mask)
    val_losses.append(val_loss)
    
    # Adjust learning rate
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_CR_gt.pt')
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        
        
    
    # Print progress
    if epoch % 10 == 0:
        test_loss  = evaluate_tr(data.test_mask)
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Test Loss: {test_loss:.4f}, ')
        
    # Check for early stopping
    if early_stop_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}")
        break

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()


In [ ]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_CR_gt.pt'))
test_loss = evaluate_tr(data.test_mask)
print(f'Final Test Results - Loss: {test_loss:.4f}')

In [ ]:
# Generate predictions for all nodes
model.eval()
with torch.no_grad():
    predictions = model(data).squeeze().cpu().numpy()
# Add predictions back to the geodataframe for visualization
edges_gnn['predictions_tr'] = predictions
#Visualize the predictions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
# Plot the original traffic speed
edges_gnn.plot(column='traffic_speed', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('Original Traffic Speed')
ax1.set_axis_off()
# Plot the predicted traffic speed
edges_gnn.plot(column='predictions_tr', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('Predicted Traffic Speed')
ax2.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Load new data
nodes_gnn2 = gpd.read_file('nodes_CR.geojson')
edges_gnn2 = gpd.read_file('edges_CR.geojson')
#plot
f, ax = plt.subplots(figsize=(8,8))
nodes_gnn2.plot(ax=ax, color='blue', markersize=3, zorder=2)
edges_gnn2.plot(ax=ax, linewidth= 1, edgecolor= 'green', zorder=1)


In [ ]:
# Check the data
nodes_gnn2.head()

In [ ]:
# Preprocess the data and prepare the dataset
X2 = nodes_gnn2.drop(columns=['geometry', 'junction', 'highway'])
X2 = np.array(X2.values)
scaler = StandardScaler()
X2_scaled = scaler.fit_transform(X2)
X2_scaled.shape

In [ ]:
# Prepare edge index
node_to_id2 = {}
for i, node in enumerate(nodes_gnn2['osmid'].values):
    node_to_id2[node] = i
start_node_pc = [node_to_id2[i] for i in edges_gnn2['u'].values]
end_node_pc = [node_to_id2[i] for i in edges_gnn2['v'].values]
start_pc = torch.tensor(start_node_pc, dtype=torch.long)
end_pc = torch.tensor(end_node_pc, dtype=torch.long)
edge_index2 = torch.stack([start_pc, end_pc], dim=0)

Y2 = edges_gnn2['traffic_speed'].values
Y2 = np.array(Y2)

In [ ]:
# Create dataset for edge regression
data2 = CremonaDataset(edge_index2, X2_scaled, Y2)

In [ ]:
# Compare the models:
gat_model = GATv2(HIDDEN_DIM, data2[0]).to(device)
gat_model.load_state_dict(torch.load('best_CR_gat.pt'))
gt_model = GT(HIDDEN_DIM, data2[0]).to(device)
gt_model.load_state_dict(torch.load('best_CR_gt.pt'))

gat_model.eval()
gt_model.eval()
with torch.no_grad():
    gat_predictions = gat_model(data2[0]).squeeze().cpu().numpy()
    gt_predictions = gt_model(data2[0]).squeeze().cpu().numpy()
    
# Normalize the predictions
scaler = MinMaxScaler()
gat_predictions = scaler.fit_transform(gat_predictions.reshape(-1, 1))
gt_predictions = scaler.fit_transform(gt_predictions.reshape(-1, 1))

# Add predictions back to the geodataframe for visualization
edges_gnn2['gat_predictions'] = gat_predictions
edges_gnn2['gt_predictions'] = gt_predictions

# Visualize the predictions
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 6))
# Plot the original traffic speed
edges_gnn2.plot(column='traffic_speed', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('Original Traffic Speed')
ax1.set_axis_off()
# Plot the GAT predictions
edges_gnn2.plot(column='gat_predictions', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('GATv2 Predicted Traffic Speed')
ax2.set_axis_off()
# Plot the GT predictions
edges_gnn2.plot(column='gt_predictions', ax=ax3, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax3.set_title('TransformerConv Predicted Traffic Speed')
ax3.set_axis_off()
plt.tight_layout()
plt.show()

## Don't stop me now, I'm having such a good time 
### I want to try a proper transformer model, like the GPS 

In [ ]:
import argparse
import os.path as osp
from typing import Any, Dict, Optional

import torch
from torch.nn import (
    BatchNorm1d,
    Embedding,
    Linear,
    ModuleList,
    ReLU,
    Sequential,
)
from torch.optim.lr_scheduler import ReduceLROnPlateau

import torch_geometric.transforms as T
from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GPSConv, global_add_pool
from torch_geometric.nn.attention import PerformerAttention

In [ ]:
# Preprocess the data and prepare the dataset
transform = T.AddRandomWalkPE( walk_length= 20, attr_name= 'pe')
transform2 = T.AddLaplacianEigenvectorPE(k = 20, attr_name= 'pe')
data = Data(x = torch.FloatTensor(X_scaled), y = torch.FloatTensor(Y), edge_index = edge_index )
data = transform2(data)

data.num_edges

In [ ]:
 # Split data into train/val/test sets (70%/15%/15%)
np.random.seed(0)
        
n_edges = data.num_edges
indices = np.random.permutation(n_edges)
        
train_idx = indices[:int(0.7 * n_edges)]
print(len(train_idx)) #debug
val_idx = indices[int(0.7 * n_edges):int(0.85 * n_edges)]
print(len(val_idx)) #debug
test_idx = indices[int(0.85 * n_edges):]
print(len(test_idx)) #debug
        
# Create masks
train_mask = torch.zeros(n_edges, dtype=torch.bool)
val_mask = torch.zeros(n_edges, dtype=torch.bool)
test_mask = torch.zeros(n_edges, dtype=torch.bool)
    
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True
        
data.train_mask = train_mask    #size is still all nodes but only True for train nodes
data.val_mask = val_mask
data.test_mask = test_mask

data.to(device)

In [ ]:
class RedrawProjection:
    def __init__(self, model: torch.nn.Module,
                redraw_interval: Optional[int] = None):
        self.model = model
        self.redraw_interval = redraw_interval
        self.num_last_redraw = 0

    def redraw_projections(self):
        if not self.model.training or self.redraw_interval is None:
            return
        if self.num_last_redraw >= self.redraw_interval:
            fast_attentions = [
                module for module in self.model.modules()
                if isinstance(module, PerformerAttention)
            ]
            for fast_attention in fast_attentions:
                fast_attention.redraw_projection_matrix()
            self.num_last_redraw = 0
            return
        self.num_last_redraw += 1



In [ ]:
class GPS(torch.nn.Module):
    def __init__(self, channels: int, pe_dim: int, num_layers: int,
                 attn_type: str = 'performer', dropout: float = 0.0):
        super().__init__()

        self.node_emb = Linear(data.x.shape[1], channels - pe_dim)
        self.pe_lin = Linear(20, pe_dim)
        self.pe_norm = BatchNorm1d(20)

        self.convs = ModuleList()
        for _ in range(num_layers):
            nn = Sequential(
                Linear(channels, channels),
                ReLU(),
                Linear(channels, channels),
            )
            conv = GPSConv(channels, GINConv(nn), heads=4,
                           attn_type=attn_type, dropout=dropout)
            self.convs.append(conv)

        self.mlp = Sequential(
            Linear(2* channels, channels),
            ReLU(),
            Linear(channels, 1)
            
        )
        self.redraw_projection = RedrawProjection(
            self.convs,
            redraw_interval=1000 if attn_type == 'performer' else None)

    def forward(self, x, pe, edge_index):
        x_pe = self.pe_norm(pe)
        x = torch.cat((self.node_emb(x), self.pe_lin(x_pe)), 1)
        

        for conv in self.convs:
            x = conv(x, edge_index)
        
        # Edge-level prediction
        src, dst = edge_index
        x = torch.cat((x[src], x[dst]), dim=1)
        
        x = self.mlp(x)
        return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GPS(channels= 64, pe_dim=8, num_layers=2, attn_type= 'performer',
            dropout= 0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20,
                              min_lr=0.00001)

In [ ]:
model

In [ ]:
def train(data):
    model.train()
    optimizer.zero_grad()
    model.redraw_projection.redraw_projections()
    out = model(data.x, data.pe, data.edge_index)
    pred = out[data.train_mask].squeeze()
    loss = criterion(out[data.train_mask].squeeze(), data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.pe, data.edge_index)
        pred = out[mask].squeeze()
        true = data.y[mask]
        loss = criterion(pred, true)
    return loss.item()

In [ ]:
out = model(data.x, data.pe, data.edge_index)
out[data.val_mask].squeeze().size()

In [ ]:
# Training loop
EPOCHS = 500

best_loss = float('inf')
early_stop_counter = 0
early_stop_patience = 30
train_losses, val_losses = [], []

print(f"Starting training on {device}...")

for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss = train(data)
    train_losses.append(train_loss)
    
    # Evaluate on validation set
    val_loss = evaluate(data, data.val_mask)
    val_losses.append(val_loss)
    
    # Adjust learning rate
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_CR_gps.pt')
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        
        
    
    # Print progress
    if epoch % 10 == 0:
        test_loss  = evaluate(data, data.test_mask)
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Test Loss: {test_loss:.4f}, ')
        
    # Check for early stopping
    if early_stop_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}")
        break

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

In [ ]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_CR_gps.pt'))
test_loss = evaluate(data, data.test_mask)
print(f'Final Test Results - Loss: {test_loss:.4f}')

In [ ]:
# Generate predictions for all nodes
model.eval()
with torch.no_grad():
    predictions = model(data.x, data.pe, data.edge_index).squeeze().cpu().numpy()
# Add predictions back to the geodataframe for visualization
edges_gnn['gps_predictions'] = predictions
#Visualize the predictions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
# Plot the original traffic speed
edges_gnn.plot(column='traffic_speed', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('Original Traffic Speed')
ax1.set_axis_off()
# Plot the predicted traffic speed
edges_gnn.plot(column='gps_predictions', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('Predicted Traffic Speed')
ax2.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Compare the three models with the ground truth
fig, axs = plt.subplots(2, 2, figsize=(18, 10))
ax1, ax2, ax3, ax4 = axs.flatten()
# Plot the original traffic speed
edges_gnn.plot(column='traffic_speed', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('Original Traffic Speed')
ax1.set_axis_off()
# Plot the GAT predictions
edges_gnn.plot(column='gat_predictions', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('GATv2 Predicted Traffic Speed')
ax2.set_axis_off()
# Plot the GT predictions
edges_gnn.plot(column='predictions_tr', ax=ax3, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax3.set_title('TransformerConv Predicted Traffic Speed')
ax3.set_axis_off()
# Plot the GPS predictions
edges_gnn.plot(column='gps_predictions', ax=ax4, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax4.set_title('GPS Predicted Traffic Speed')
ax4.set_axis_off()
plt.tight_layout()
plt.savefig('Cremona_GNNs.png', dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
data2 = Data(x = torch.FloatTensor(X2_scaled), y = torch.FloatTensor(Y2), edge_index = edge_index2 )
data2 = transform2(data2)
#set seed
torch.manual_seed(0)
idxs = torch.randperm(data2.num_edges)
ft_idx = idxs[:int(0.3 * data2.num_edges)]
ft_val_idx = idxs[int(0.3 * data2.num_edges):int(0.4 * data2.num_edges)]

data2.train_mask = torch.zeros(data2.num_edges, dtype=torch.bool)
data2.train_mask[ft_idx] = True
data2.val_mask = torch.zeros(data2.num_edges, dtype=torch.bool)
data2.val_mask[ft_val_idx] = True





In [ ]:
# predictions for the new data
model.eval()
with torch.no_grad():
    predictions_pc = model(data2.x, data2.pe, data2.edge_index).squeeze().cpu().numpy()
    
# Normalize the predictions
scaler = MinMaxScaler()
predictions_pc = scaler.fit_transform(predictions_pc.reshape(-1, 1))
# Add predictions back to the geodataframe for visualization
edges_gnn2['gps_predictions'] = predictions_pc
#Visualize the predictions
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
# Plot the predicted traffic speed
edges_gnn2.plot(column='gps_predictions', ax=ax, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax.set_title('Predicted Traffic Speed')
ax.set_axis_off()

In [ ]:
# compare the three models
fig, axs = plt.subplots(2, 2, figsize=(14, 6))
ax1, ax2, ax3, ax4 = axs.flatten()
# Plot the GAT predictions
edges_gnn2.plot(column='gat_predictions', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('GATv2 Predicted Traffic Speed')
ax1.set_axis_off()
# Plot the GT predictions
edges_gnn2.plot(column='gt_predictions', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('TransformerConv Predicted Traffic Speed')
ax2.set_axis_off()
# Plot the GPS predictions
edges_gnn2.plot(column='gps_predictions', ax=ax3, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax3.set_title('GPS Predicted Traffic Speed')
ax3.set_axis_off()
# Plot the original traffic speed
edges_gnn2.plot(column='traffic_speed', ax=ax4, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax4.set_title('Original Traffic Speed')
ax4.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Let's try to finetune the GPS model a bit more on the new data
# Load best model for final evaluation
model.load_state_dict(torch.load('best_CR_gps.pt'))

In [ ]:
# Training loop
EPOCHS = 100
best_loss = float('inf')
early_stop_counter = 0
early_stop_patience = 30
train_losses, val_losses = [], []
print(f"Starting training on {device}...")
for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss = train(data2)
    train_losses.append(train_loss)
    
    # Evaluate on validation set
    val_loss = evaluate(data2, data2.val_mask)
    val_losses.append(val_loss)
    
    # Adjust learning rate
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_gps_finetune.pt')
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        
        
    
    # Print progress
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, ')
        
    # Check for early stopping
    if early_stop_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}")
        break

In [ ]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_gps_finetune.pt'))

# Generate predictions for all nodes to compare finetuned model with the previous one
model.eval()
with torch.no_grad():
    predictions_pc = model(data2.x, data2.pe, data2.edge_index).squeeze().cpu().numpy()
# Normalize the predictions
scaler = MinMaxScaler()
predictions_pc = scaler.fit_transform(predictions_pc.reshape(-1, 1))
# Add predictions back to the geodataframe for visualization
edges_gnn2['gps_predictions_finetune'] = predictions_pc
#Compare the two models with the ground truth
fig, axs = plt.subplots(1, 3, figsize=(14, 6))
ax1, ax2, ax3 = axs.flatten()
# Plot the original traffic speed
edges_gnn2.plot(column='traffic_speed', ax=ax1, legend=True, cmap='viridis', legend_kwds={
    'label': "Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax1.set_title('Original Traffic Speed')
ax1.set_axis_off()
# Plot the GPS predictions
edges_gnn2.plot(column='gps_predictions', ax=ax2, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax2.set_title('GPS Predicted Traffic Speed')
ax2.set_axis_off()
# Plot the finetuned GPS predictions
edges_gnn2.plot(column='gps_predictions_finetune', ax=ax3, legend=True, cmap='viridis', legend_kwds={
    'label': "Predicted Traffic Speed (km/h)",
    'orientation': "horizontal"
})
ax3.set_title('Finetuned GPS Predicted Traffic Speed')
ax3.set_axis_off()
plt.tight_layout()
# Save the picture
plt.savefig('Cremona_GPS_finetune.png', dpi=600, bbox_inches='tight')
plt.show()

